# FastText Batch Training in Google Colab

This notebook demonstrates how to:
1. Clone a Git repository from a specific branch
2. Install dependencies from `pyproject.toml`
3. Authenticate with Google Cloud
4. Connect to Google Cloud Storage (GCS) bucket
5. Run batch training with checkpoint management

## Prerequisites
- Google Cloud Project with $300 free credit
- GCS bucket: `gs://multi-index-rag-bucket/data`
- Git repository: GitHub (set your URL below)

## Session Duration
- Free tier: 12 hours maximum per session
- High-RAM: 30 hours maximum per session
- Recommended: High-RAM runtime for batch training

## Section 1: Import Required Libraries

In [ ]:
import subprocess
import os
import sys
import json
import logging
from pathlib import Path

# Colab-specific imports
try:
    from google.colab import auth, drive
    from google.colab import output as colab_output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"Running in Colab: {IN_COLAB}")
print(f"Python version: {sys.version}")

## Section 2: Configuration Variables

Set your Git repository details and GCS bucket here.

In [ ]:
# ============================================================================
# CONFIGURATION: Modify these variables for your setup
# ============================================================================

# Git repository settings
GIT_REPO_URL = "https://github.com/yourusername/multi-index-rag.git"  # Change this!
GIT_BRANCH = "batch-training"  # Branch to checkout
REPO_LOCAL_PATH = "/content/multi-index-rag"  # Where to clone

# Google Cloud settings
GCS_BUCKET_NAME = "multi-index-rag-bucket"  # Change to your bucket name
GCS_PROJECT_ID = "your-project-id"  # Change to your GCP project ID

# Training settings
TREE_TYPE = "SLT"  # SLT, OPT, or SLT-TYPE
BATCH_SIZE = 30000  # Formulas per batch (adjust based on available RAM)
EPOCHS = 5  # Training epochs per batch
NUM_WORKERS = 4  # Parallel workers (adjust for CPU count in Colab: usually 2)

# ============================================================================

logger.info("Configuration loaded:")
logger.info(f"  Git repo: {GIT_REPO_URL}")
logger.info(f"  Branch: {GIT_BRANCH}")
logger.info(f"  GCS bucket: gs://{GCS_BUCKET_NAME}")
logger.info(f"  Training tree type: {TREE_TYPE}")
logger.info(f"  Batch size: {BATCH_SIZE}")

## Section 3: Checkout Git Repository from Specific Branch

In [ ]:
# Clone repository from specific branch
repo_path = Path(REPO_LOCAL_PATH)

if repo_path.exists():
    logger.info(f"Repository already exists at {REPO_LOCAL_PATH}")
    logger.info("Pulling latest changes...")
    subprocess.run(
        ["git", "-C", REPO_LOCAL_PATH, "pull", "origin", GIT_BRANCH],
        check=True
    )
else:
    logger.info(f"Cloning repository from {GIT_REPO_URL}")
    subprocess.run(
        ["git", "clone", "--branch", GIT_BRANCH, GIT_REPO_URL, REPO_LOCAL_PATH],
        check=True
    )
    logger.info(f"✓ Repository cloned to {REPO_LOCAL_PATH}")

# Verify branch
result = subprocess.run(
    ["git", "-C", REPO_LOCAL_PATH, "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True,
    text=True
)
current_branch = result.stdout.strip()
logger.info(f"✓ Current branch: {current_branch}")

# Show commit info
result = subprocess.run(
    ["git", "-C", REPO_LOCAL_PATH, "log", "-1", "--oneline"],
    capture_output=True,
    text=True
)
logger.info(f"Latest commit: {result.stdout.strip()}")

## Section 4: Install Dependencies from pyproject.toml

In [ ]:
import time

logger.info("Installing dependencies from pyproject.toml...")
logger.info("This may take 5-10 minutes. Please wait...")

start_time = time.time()

# Install poetry if not available
try:
    subprocess.run(["poetry", "--version"], capture_output=True, check=True)
    logger.info("✓ Poetry already installed")
except Exception:
    logger.info("Installing poetry...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "poetry"],
        check=True
    )

# Install dependencies using poetry
try:
    result = subprocess.run(
        ["poetry", "install", "--no-root"],
        cwd=REPO_LOCAL_PATH,
        capture_output=True,
        text=True,
        timeout=600  # 10 minute timeout
    )
    
    if result.returncode == 0:
        logger.info("✓ Poetry install successful")
    else:
        logger.warning(f"Poetry stderr: {result.stderr}")
        # Fallback to pip if poetry fails
        logger.info("Falling back to pip install...")
        pyproject_path = Path(REPO_LOCAL_PATH) / "pyproject.toml"
        if pyproject_path.exists():
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-e", REPO_LOCAL_PATH],
                check=True,
                timeout=600
            )
            logger.info("✓ Pip install successful")
        
except subprocess.TimeoutExpired:
    logger.error("Installation timed out after 10 minutes")
except Exception as e:
    logger.error(f"Installation error: {e}")

elapsed_time = time.time() - start_time
logger.info(f"Installation completed in {elapsed_time:.1f} seconds")

# Verify key packages installed
logger.info("\nVerifying key packages...")
try:
    import gensim
    logger.info(f"✓ gensim {gensim.__version__}")
except ImportError:
    logger.warning("⚠ gensim not found")

try:
    import pandas as pd
    logger.info(f"✓ pandas {pd.__version__}")
except ImportError:
    logger.warning("⚠ pandas not found")

try:
    from google.cloud import storage
    logger.info("✓ google-cloud-storage installed")
except ImportError:
    logger.warning("⚠ google-cloud-storage not found - installing...")

## Section 5: Authenticate and Connect to Google Cloud Bucket

**Note:** You will be prompted to authenticate with your Google account. Click the link and follow the authorization flow.

In [ ]:
if IN_COLAB:
    logger.info("Authenticating with Google Cloud...")
    try:
        auth.authenticate_user()
        logger.info("✓ Google Cloud authentication successful")
    except Exception as e:
        logger.error(f"Authentication failed: {e}")
        logger.info("Please run this cell again and complete the authorization flow")
else:
    logger.info("Not in Colab. Skipping Colab-specific authentication.")
    logger.info("For local testing, use: gcloud auth application-default login")

# Install google-cloud-storage if needed
try:
    from google.cloud import storage
    logger.info("google-cloud-storage already installed")
except ImportError:
    logger.info("Installing google-cloud-storage...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "google-cloud-storage"],
        check=True
    )
    from google.cloud import storage
    logger.info("✓ google-cloud-storage installed")

# Initialize storage client
logger.info(f"\nInitializing storage client for bucket: gs://{GCS_BUCKET_NAME}")
try:
    storage_client = storage.Client(project=GCS_PROJECT_ID)
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    
    logger.info(f"✓ Connected to bucket: {bucket.name}")
    logger.info(f"  Bucket location: {bucket.location}")
    logger.info(f"  Bucket storage class: {bucket.storage_class}")
    
except Exception as e:
    logger.error(f"Failed to connect to bucket: {e}")
    logger.info("Make sure your GCS_BUCKET_NAME and GCS_PROJECT_ID are correct")

## Section 6: Verify Connection and List Bucket Contents

In [ ]:
try:
    logger.info(f"Listing contents of gs://{GCS_BUCKET_NAME}/...")
    
    blobs = storage_client.list_blobs(GCS_BUCKET_NAME, max_results=20)
    
    file_count = 0
    dir_prefixes = set()
    
    for blob in blobs:
        file_count += 1
        # Extract directory prefixes
        if "/" in blob.name:
            prefix = blob.name.split("/")[0]
            dir_prefixes.add(prefix)
    
    logger.info(f"✓ Top-level directories found: {', '.join(sorted(dir_prefixes))}")
    logger.info(f"✓ Sample files: {file_count} items shown")
    
    # Check for specific directories needed for batch training
    required_dirs = ["data/raw/collection/formula/latex_representation_v3", "data/formula-indexing/fasttext"]
    
    logger.info("\nChecking for required directories:")
    for req_dir in required_dirs:
        blobs_in_dir = list(storage_client.list_blobs(GCS_BUCKET_NAME, prefix=req_dir, max_results=1))
        if blobs_in_dir:
            logger.info(f"  ✓ {req_dir}/ exists")
        else:
            logger.info(f"  ⚠ {req_dir}/ not found (will be created)")
    
    logger.info("\n✓ Connection verified! Bucket is accessible.")
    
except Exception as e:
    logger.error(f"Error listing bucket contents: {e}")
    logger.info("Please check:")
    logger.info(f"  - GCS_BUCKET_NAME: {GCS_BUCKET_NAME}")
    logger.info(f"  - GCS_PROJECT_ID: {GCS_PROJECT_ID}")
    logger.info(f"  - Your Google Cloud permissions")

## Section 7: Setup Paths and Import Training Module

In [ ]:
# Add repository to Python path
sys.path.insert(0, REPO_LOCAL_PATH)

logger.info("Importing training module...")
try:
    from multirag.embedding.formula_trainer import FormulaTrainer
    logger.info("✓ FormulaTrainer imported successfully")
except ImportError as e:
    logger.error(f"Failed to import FormulaTrainer: {e}")
    logger.info("Make sure the repository is properly installed")
    raise

# Verify that path configs point to GCS (if needed)
logger.info("\nPath configuration check:")
try:
    from multirag.config.path_configs import LATEX_REPRESENTATION, FASTTEXT_MODEL_DIR
    logger.info(f"  LATEX_REPRESENTATION: {LATEX_REPRESENTATION}")
    logger.info(f"  FASTTEXT_MODEL_DIR: {FASTTEXT_MODEL_DIR}")
    
    # Note: In Colab, you may need to update these to use GCS paths (gs://...)
    # See: BATCH_TRAINING_GUIDE.md for details
    
except Exception as e:
    logger.warning(f"Could not load path configs: {e}")

## Section 8: Run Batch Training

This cell runs one batch of training. The checkpoint is automatically saved after each batch, allowing you to resume from the next cell in a new Colab session.

In [ ]:
logger.info(f"\nStarting batch training ({TREE_TYPE})...")
logger.info(f"Batch size: {BATCH_SIZE} formulas, Epochs: {EPOCHS}, Workers: {NUM_WORKERS}\n")

try:
    # Initialize trainer
    trainer = FormulaTrainer(
        tree_type=TREE_TYPE,
        num_workers=NUM_WORKERS,
        vector_size=200,
        window=5,
    )
    
    # Load checkpoint to see current progress
    checkpoint = trainer._load_checkpoint()
    batches_completed = checkpoint['batches_completed']
    total_processed = checkpoint['total_formulas_processed']
    total_available = checkpoint['total_formulas_available']
    
    logger.info("=" * 70)
    logger.info("CHECKPOINT STATUS")
    logger.info("=" * 70)
    logger.info(f"Batches completed: {batches_completed}")
    logger.info(f"Total formulas processed: {total_processed}")
    if total_available > 0:
        progress = total_processed / total_available * 100
        logger.info(f"Progress: {progress:.1f}% ({total_processed}/{total_available})")
    logger.info("=" * 70 + "\n")
    
    # Check if training is complete
    if total_available > 0 and total_processed >= total_available:
        logger.info("✓ All formulas already processed! Training complete.")
    else:
        # Run batch training
        logger.info(f"Running batch #{batches_completed + 1}...\n")
        model = trainer.train_batch(
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            formula_column="formula"
        )
        
        # Show updated checkpoint
        checkpoint = trainer._load_checkpoint()
        logger.info("\n" + "=" * 70)
        logger.info("BATCH COMPLETE")
        logger.info("=" * 70)
        logger.info(f"Batch #{checkpoint['batches_completed']} finished")
        logger.info(f"Formulas processed: {checkpoint['total_formulas_processed']}")
        if checkpoint['total_formulas_available'] > 0:
            progress = checkpoint['total_formulas_processed'] / checkpoint['total_formulas_available']
            logger.info(f"Progress: {progress*100:.1f}%")
        logger.info("=" * 70)
        logger.info("\n✓ Checkpoint saved. You can safely disconnect.")
        logger.info("  To continue training, run the next batch cell.")
    
except KeyboardInterrupt:
    logger.info("\n⚠ Training interrupted by user. Checkpoint saved.")
except Exception as e:
    logger.error(f"Error during training: {e}")
    logger.info("Checkpoint has been saved. Please check the error and try again.")

## Section 9: Check Training Progress

In [ ]:
# Run this cell anytime to check progress without training
from multirag.embedding.formula_trainer import FormulaTrainer

trainer = FormulaTrainer(tree_type=TREE_TYPE)
checkpoint = trainer._load_checkpoint()

print("\n" + "="*70)
print("TRAINING PROGRESS REPORT")
print("="*70)
print(f"Tree type: {checkpoint['tree_type']}")
print(f"Batches completed: {checkpoint['batches_completed']}")
print(f"Total formulas processed: {checkpoint['total_formulas_processed']}")
print(f"Total formulas available: {checkpoint['total_formulas_available']}")
print(f"Completed files: {len(checkpoint['completed_files'])} files")
print(f"Current file: {checkpoint['last_file_number']}")
print(f"Current row index: {checkpoint['last_file_row_index']}")

if checkpoint['total_formulas_available'] > 0:
    progress = checkpoint['total_formulas_processed'] / checkpoint['total_formulas_available']
    print(f"\n▓ Progress: {progress*100:.1f}%")
    
    # Progress bar
    bar_length = 40
    filled = int(bar_length * progress)
    bar = "█" * filled + "░" * (bar_length - filled)
    print(f"  [{bar}]")
    
    remaining = checkpoint['total_formulas_available'] - checkpoint['total_formulas_processed']
    if remaining > 0:
        print(f"\n  Remaining: {remaining:,} formulas")
        if checkpoint['batches_completed'] > 0:
            avg_per_batch = checkpoint['total_formulas_processed'] / checkpoint['batches_completed']
            batches_remaining = remaining / avg_per_batch
            print(f"  Estimated batches remaining: ~{batches_remaining:.1f}")
else:
    print("\n⚠ Total formulas not yet determined")

print("="*70)

## Notes and Troubleshooting

### Running Multiple Batches

To train in a loop until all formulas are processed:

```python
from multirag.embedding.formula_trainer import FormulaTrainer

trainer = FormulaTrainer(tree_type="SLT")

while True:
    checkpoint = trainer._load_checkpoint()
    if checkpoint['total_formulas_available'] > 0:
        if checkpoint['total_formulas_processed'] >= checkpoint['total_formulas_available']:
            print("✓ All formulas processed!")
            break
    
    # Run batch
    model = trainer.train_batch(batch_size=30000, epochs=5)
    print(f"✓ Batch {checkpoint['batches_completed'] + 1} complete")
```

### Common Issues

**1. Out of Memory (OOM)**
- Reduce `BATCH_SIZE` (try 10000-15000)
- Use high-RAM runtime in Colab
- Reduce `NUM_WORKERS` to 2

**2. Permission Denied on GCS**
- Check that your Google account has access to the bucket
- Verify bucket name and project ID
- Use `gsutil` to test: `gsutil ls gs://your-bucket/`

**3. Dependencies not installing**
- Try manual install: `pip install gensim pandas google-cloud-storage`
- Check internet connection
- Use `pip install --upgrade pip` first

**4. Training stuck on same batch**
- Delete checkpoint: `Path(...checkpoint.json).unlink()`
- Check LaTeX formula validity
- Review logs for specific errors

### Monitoring Tips

- **Memory usage:** Watch the Memory indicator in Colab top-right
- **Execution time:** Session times out after 12 hours (free) or 30 hours (high-RAM)
- **Disk space:** Check `/content` available space with `!df -h`

### Resource Estimates

For complete 100K formula dataset with 30K batches:
- **Time:** ~10 hours (5 epochs × 4 batches)
- **Storage (GCS):** ~50 MB (model) + 100 MB (corpus)
- **Cost:** <$1 (mostly free tier)

### Further Reading

- See `BATCH_TRAINING_GUIDE.md` in the repository for detailed documentation
- See `colab_batch_trainer.py` for standalone script example

## Appendix: Quick Reference

### Session Workflow

1. **First Session:**
   - Run all cells from Section 1-7 (setup)
   - Run Section 8 (batch training)
   - Check progress with Section 9

2. **Subsequent Sessions:**
   - Run Sections 1-7 (setup - takes ~5 min)
   - Run Section 8 (automatically resumes)
   - Check progress with Section 9

3. **If Colab Times Out:**
   - Click "Reconnect" → Run all sections again
   - Training auto-resumes from checkpoint

### Useful Google Cloud Commands

```bash
# Create bucket
gsutil mb gs://multi-index-rag-bucket

# Copy local data to GCS
gsutil -m cp -r ~/Documents/projects/multi-index-rag/data/* gs://multi-index-rag-bucket/data/

# List bucket contents
gsutil ls gs://multi-index-rag-bucket/data/

# Download model after training
gsutil cp gs://multi-index-rag-bucket/data/formula-indexing/fasttext/*.bin ~/Downloads/
```

### GCS Paths Structure

```
gs://multi-index-rag-bucket/
├── data/
│   ├── raw/
│   │   └── collection/
│   │       └── formula/
│   │           └── latex_representation_v3/  (TSV files: 1.tsv, 2.tsv, ...)
│   ├── formula-indexing/
│   │   ├── encoder_maps_slt.tsv
│   │   ├── encoder_maps_opt.tsv
│   │   └── fasttext/
│   │       ├── fasttext_model_slt.bin
│   │       ├── corpus_slt.txt
│   │       └── checkpoint_slt.json  ← This tracks progress!
```

### Import Modules in Notebooks

```python
from multirag.embedding.formula_trainer import FormulaTrainer
from multirag.config.path_configs import LATEX_REPRESENTATION, FASTTEXT_MODEL_DIR
from multirag.formula_search import SLTGenerator, OPTGenerator
```